# 0 Préparation connexion et nettoyage

In [18]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

if es.indices.exists_alias(name="products"):
    es.indices.delete_alias(index="*", name="products")

#es.indices.delete(index="products_v1")
#es.indices.delete(index="products_v2")

if es.indices.exists(index="orders"):
    es.indices.delete(index="orders")

print("prêt pour le lab")

NotFoundError: NotFoundError(404, 'index_not_found_exception', 'no such index [products_v2]', products_v2, index_or_alias)

# 1 créer index "products" avec mapping multi-type

In [8]:
products_mapping = {
    "mappings":{
        "properties": {
            "name": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}}
            },
            "category": {"type": "keyword"},
            "price": {"type": "float"},
            "stock": {"type": "integer"},
            "tags": {"type": "keyword"},
            "created_at": {"type": "date"}
        }
    }
}

es.indices.create(index="products", body=products_mapping)

print("Index 'products' créé")

Index 'products' créé


# 2 Buls insert: plusieurs documents

In [9]:
from datetime import datetime
from elasticsearch.helpers import bulk

products = [
    {"_index": "products", "_id": 1, "_source": {"name": "iPhone is Pro", "category": "smartphone", "price": 1200, "stock": 10, "tags": ["apple", "mobile"], "created_at": datetime.now()}},
    {"_index": "products", "_id": 2, "_source": {"name": "Samsung galaxy s23", "category": "smartphone", "price": 900, "stock": 18, "tags": ["samsung", "android"], "created_at": datetime.now()}},
    {"_index": "products", "_id": 3, "_source": {"name": "Dell XPS 13", "category": "laptop", "price": 1500, "stock": 5, "tags": ["dell", "laptop"], "created_at": datetime.now()}},
]

bulk(es, products)
print("Bulk insert terminé")

Bulk insert terminé


# 3 Recherche smple et filtrée

In [10]:
res = es.search(index="products", query={"match": {"name": "iPhone"}})
print("Recherche text : ", [hit["_source"] for hit in res["hits"]["hits"]])

res = es.search(index="products", query={"term": {"category": "smartphone"}})
print("Filtre exact: ", [hit["_source"] for hit in res["hits"]["hits"]])

Recherche text :  [{'name': 'iPhone is Pro', 'category': 'smartphone', 'price': 1200, 'stock': 10, 'tags': ['apple', 'mobile'], 'created_at': '2026-04-02T15:16:45.719097'}]
Filtre exact:  [{'name': 'iPhone is Pro', 'category': 'smartphone', 'price': 1200, 'stock': 10, 'tags': ['apple', 'mobile'], 'created_at': '2026-04-02T15:16:45.719097'}, {'name': 'Samsung galaxy s23', 'category': 'smartphone', 'price': 900, 'stock': 18, 'tags': ['samsung', 'android'], 'created_at': '2026-04-02T15:16:45.719102'}]


# 4 Aggregation : stats sur les prix

In [11]:
res = es.search(
    index="products",
    size=0,
    aggs={
        "avg_price": {"avg": {"field": "price"}},
        "max_price": {"max": {"field": "price"}},
        "min_price": {"min": {"field": "price"}}
    }
)

print("Aggregation des prix : ", res['aggregations'])

Aggregation des prix :  {'avg_price': {'value': 1200.0}, 'max_price': {'value': 1500.0}, 'min_price': {'value': 900.0}}


# 5 Update partiel

In [12]:
es.update(index="products", id=1, doc={"doc": {"stock": 8}})
print("Sock mis à jour")

Sock mis à jour


# 6 Alias pour version future

In [13]:
es.indices.put_alias(index="products", name="products_current")
print("Alias créé")

Alias créé


# 7 Index Template: logs ou produits futurs

In [14]:
template = {
    "index_patterns": ["products.*"],
    "template": {
        "settings": {"number_of_shards": 1},
        "mappings": {
            "properties": {
                "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                "category": {"type": "keyword"},
                "price": {"type": "float"},
                "created_at": {"type": "date"}
            }
        },

        "aliases": {"products_current": {}}
    }
}

es.indices.put_index_template(name="products_template", body=template)
print("Template créé")

Template créé


# 8 Reindex (mise à jour version v2)

In [19]:
mapping_v2 = {
    "mappings": {
        "properties":{
            "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
            "category": {"type": "keyword"},
            "price": {"type": "float"},
            "stock": {"type": "integer"},
            "discount": {"type": "float"}
        } 
    }
}

es.indices.create(index="products_v2", body=mapping_v2)

es.reindex(body={"source": {"index": "products"}, "dest": {"index": "products_v2"}})
print("Reindex v2 terminé")

Reindex v2 terminé


# switch alias (production sans downtime)

In [20]:
actions = [
    {"remove": {"index": "products", "alias": "products_current"}},
    {"add": {"index": "products_v2", "alias": "products_current"}}
]

es.indices.update_aliases(body={"actions": actions})
print("Alias switch vers v2")

Alias switch vers v2


# 10 Recherche avancée

In [21]:
res = es.search(index="products_current", query={
    "bool": {
        "must": [{"term": {"category": "smartphone"}}],
        "filter": [{"range": {"price": {"gt": 1000}}}]
    }
})

print("smartphones > 1000$ : ", [hit["_source"] for hit in res["hits"]["hits"]])

smartphones > 1000$ :  [{'name': 'iPhone is Pro', 'category': 'smartphone', 'price': 1200, 'stock': 10, 'tags': ['apple', 'mobile'], 'created_at': '2026-04-02T15:16:45.719097', 'doc': {'stock': 8}}]
